# BSM L06F — Colab backbone for face enrollment

This notebook is a real training workflow for the shared tiny CNN backbone.

What you need to produce:
- a tiny backbone trained on a real face dataset
- a TFLite export named `tiny_face_backbone.tflite`
- a labels file named `tiny_face_labels.txt`
- the files placed into `student/apps/lesson_f_app/app/src/main/assets/`

Android side contract:
- input: `96x96x3`
- embedding size: `32`
- on-device fine-tuning target: `head` only


## 1. Install and imports
Run this in Colab. It installs the packages needed for dataset loading, training, and export.


In [ ]:
!pip -q install tensorflow-datasets
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

INPUT_SHAPE = (96, 96, 3)
EMBEDDING_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 12
STUDENT_ID = '123456'


## 2. Download dataset
Use a real dataset with face crops and identity labels. For a deterministic classroom flow, keep the dataset size small enough to train quickly.

The cell below uses a public dataset loader as a template. If you choose another dataset source, keep the preprocessing contract identical.


In [ ]:
# Example loader. Replace with the final face dataset if your class uses a different source.
# The important part is to end up with (image, label) pairs and fixed label order.
dataset_name = 'lfw'
builder = tfds.builder(dataset_name)
builder.download_and_prepare()
train_ds_raw = tfds.load(dataset_name, split='train[:80%]', as_supervised=True)
val_ds_raw = tfds.load(dataset_name, split='train[80%:]', as_supervised=True)


## 3. Preprocess
Resize to `96x96`, normalize to `[0, 1]`, and keep the label mapping stable.


In [ ]:
class_names = builder.info.features['label'].names
num_classes = len(class_names)

def preprocess(image, label):
    image = tf.image.resize(image, INPUT_SHAPE[:2])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_ds_raw.map(preprocess).shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


## 4. Model architecture
Build the tiny CNN backbone and the classification head. The backbone output is the embedding that Android will fine-tune around.


In [ ]:
def build_backbone(input_shape=INPUT_SHAPE, embedding_size=EMBEDDING_SIZE):
    inputs = keras.Input(shape=input_shape, name='face_input')
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embedding_size, activation='relu', name='embedding')(x)
    return keras.Model(inputs, x, name='tiny_face_backbone')

def build_classifier(num_classes):
    inputs = keras.Input(shape=INPUT_SHAPE, name='face_input')
    backbone = build_backbone()
    x = backbone(inputs)
    outputs = layers.Dense(num_classes, activation='softmax', name='identity_head')(x)
    return keras.Model(inputs, outputs, name='tiny_face_classifier')

model = build_classifier(num_classes)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. Train
Run training in Colab. This is the step students should actually execute after replacing the dataset source with the one used in the course.


In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)


## 6. Export backbone
Export the backbone only. Android will load `tiny_face_backbone.tflite` and fine-tune the small head locally.


In [ ]:
backbone_only = keras.Model(model.input, model.get_layer('tiny_face_backbone').output, name='tiny_face_backbone_export')
converter = tf.lite.TFLiteConverter.from_keras_model(backbone_only)
tflite_model = converter.convert()
with open('tiny_face_backbone.tflite', 'wb') as f:
    f.write(tflite_model)
with open('tiny_face_labels.txt', 'w') as f:
    f.write('
'.join(class_names))


## 7. Download finished model
After training finishes, use this cell to download the exported files. In Colab it will trigger a download. In plain Jupyter it will show file links.


In [ ]:
try:
    from google.colab import files
    files.download('tiny_face_backbone.tflite')
    files.download('tiny_face_labels.txt')
except Exception:
    from IPython.display import FileLink, display
    display(FileLink('tiny_face_backbone.tflite'))
    display(FileLink('tiny_face_labels.txt'))


## 8. Where to put the files
Copy the exported files into the Android app:
- `student/apps/lesson_f_app/app/src/main/assets/tiny_face_backbone.tflite`
- `student/apps/lesson_f_app/app/src/main/assets/tiny_face_labels.txt`

The Android runtime contract in the app already points to those paths.
